# Sesión 3 · Automatización del cierre mensual

**Pregunta de negocio:** *¿Cómo pasamos de 3 días de copiar y pegar a un cierre que se ejecuta con un comando, se valida solo y deja en la bandeja de cada director de canal su informe?*

> 🔑 **Versión con soluciones.**

| Bloque | Contenido | Min |
|---|---|---|
| 0 | Demo del resultado final | 5 |
| 1 | Ficheros: leer 12 Excel con formatos distintos | 25 |
| 2 | Validación y control | 20 |
| 3 | Real vs presupuesto (con reparto de gastos) | 20 |
| 4 | Salidas: Excel formateado, PDF y correos | 20 |
| 5 | Del notebook al proceso: script, configuración, Git y programación | 15 |

**Los datos de entrada** (carpeta `data/entrada/`):
- `bordereaux/siniestros_2025-MM.xlsx`: un Excel por mes con los siniestros declarados. **En julio cambió el sistema** y el formato es distinto.
- `presupuesto_2025.xlsx`: presupuesto por canal, concepto y mes (formato ancho, como en Excel).
- `gastos_estructura_2025.xlsx`: gastos reales por mes, centro de coste y partida.
- `parametros/parametros_2025.xlsx`: comisiones por canal y claves de reparto de gastos.

In [1]:
# @title ⚙️ Ejecuta esta celda primero (prepara el entorno) { display-mode: "form" }
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SSerraPe/curso-python-finanzas.git"
COPIA_DRIVE = Path("/content/drive/MyDrive/curso-python-finanzas")  # la crea el notebook 00
EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("⚠️ No se ha podido conectar Google Drive:", e)
    if COPIA_DRIVE.is_dir():
        os.chdir(COPIA_DRIVE)
    else:
        print("⚠️ No encuentro tu copia del curso en Mi unidad › curso-python-finanzas (ver notebook 00).")
        print("   Uso una copia TEMPORAL: lo que generes se pierde al cerrar Colab.")
        if not Path("/content/curso-python-finanzas").exists():
            subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/curso-python-finanzas"], check=True)
        os.chdir("/content/curso-python-finanzas")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    # En tu ordenador: nos situamos en la carpeta raíz del curso (la que tiene data/ y src/)
    while not (Path("data").is_dir() and Path("src").is_dir()) and Path.cwd() != Path.cwd().parent:
        os.chdir("..")

sys.path.insert(0, str(Path("src").resolve()))
print("✅ Entorno listo. Carpeta de trabajo:", Path.cwd())

✅ Entorno listo. Carpeta de trabajo: /content/curso-python-finanzas


In [2]:
import logging
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from curso_seguros.datos import DATA, OUTPUT, cargar_cartera
from curso_seguros.comprobar import comprobar

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 30)

---
# Bloque 0 · Demo: el cierre en un comando

Este es el **producto final** de hoy. Un script lee todo, valida, calcula y genera los entregables. Ejecútalo:

In [3]:
r = subprocess.run([sys.executable, "scripts/run_report.py", "--periodo", "2025-09"],
                   capture_output=True, text=True)
print(r.stderr[-1500:] or r.stdout[-1500:])

2026-10-08 16:09:13,521 | INFO | === Cierre 2025-09 ===
2026-10-08 16:09:13,969 | INFO | Bordereau 01: 708 siniestros, 1127711.41 €
2026-10-08 16:09:14,127 | INFO | Bordereau 02: 1395 siniestros, 3077082.95 €
2026-10-08 16:09:14,344 | INFO | Bordereau 03: 2007 siniestros, 7796494.64 €
2026-10-08 16:09:14,603 | INFO | Bordereau 04: 2316 siniestros, 4634024.55 €
2026-10-08 16:09:14,915 | INFO | Bordereau 05: 2690 siniestros, 6858504.88 €
2026-10-08 16:09:15,285 | INFO | Bordereau 06: 2686 siniestros, 4985795.65 €
2026-10-08 16:09:15,721 | INFO | Bordereau 07: 2899 siniestros, 5816172.66 €
2026-10-08 16:09:16,053 | INFO | Bordereau 08: 2668 siniestros, 6632866.34 €
2026-10-08 16:09:16,366 | INFO | Bordereau 09: 2433 siniestros, 4518530.68 €
2026-10-08 16:09:17,433 | INFO | Controles: 15 KO de 63
2026-10-08 16:09:17,536 | INFO | Excel generados en /content/curso-python-finanzas/output/2025-09
2026-10-08 16:09:18,061 | INFO | PDF generado: cierre_2025-09.pdf
2026-10-08 16:09:18,071 | INFO |

In [4]:
for f in sorted((OUTPUT / "2025-09").rglob("*.*")):
    print(f.relative_to(OUTPUT))

2025-09/cierre_2025-09.pdf
2025-09/cierre_2025-09.xlsx
2025-09/correos/correo_Agentes.eml
2025-09/correos/correo_Bancaseguros.eml
2025-09/correos/correo_Corredores.eml
2025-09/correos/correo_Directo.eml
2025-09/ejecucion.log
2025-09/por_canal/cierre_2025-09_Agentes.xlsx
2025-09/por_canal/cierre_2025-09_Bancaseguros.xlsx
2025-09/por_canal/cierre_2025-09_Corredores.xlsx
2025-09/por_canal/cierre_2025-09_Directo.xlsx


> En Colab puedes descargar cualquier fichero desde el panel 📁 de la izquierda (clic derecho → Descargar). Abre el Excel y el PDF para ver qué vamos a construir.

Ahora lo construimos **pieza a pieza**.

---
# Bloque 1 · Manipulación de ficheros

### `pathlib`: rutas que funcionan en Windows, Mac y Colab

In [5]:
carpeta = DATA / "entrada" / "bordereaux"          # el operador / une rutas
ficheros = sorted(carpeta.glob("siniestros_2025-*.xlsx"))
for f in ficheros:
    print(f.name, f"{f.stat().st_size / 1024:,.0f} KB")

siniestros_2025-01.xlsx 36 KB
siniestros_2025-02.xlsx 66 KB
siniestros_2025-03.xlsx 91 KB
siniestros_2025-04.xlsx 105 KB
siniestros_2025-05.xlsx 121 KB
siniestros_2025-06.xlsx 121 KB
siniestros_2025-07.xlsx 124 KB
siniestros_2025-08.xlsx 116 KB
siniestros_2025-09.xlsx 106 KB
siniestros_2025-10.xlsx 100 KB
siniestros_2025-11.xlsx 84 KB
siniestros_2025-12.xlsx 66 KB


### ¿Cómo es un fichero por dentro?
Leemos **sin cabecera** (`header=None`) para ver el fichero tal cual:

In [6]:
pd.read_excel(ficheros[0], sheet_name="Detalle", header=None).head(6)

,0,1,2,3,4,5,6,7
0,Bordereau de siniestros declarados - enero 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fuente: Sistema de Siniestros (extracción auto...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nº Siniestro,Nº Póliza,Fecha ocurrencia,Fecha declaración,Canal,Región,Importe (€),Estado
4,S25-000001,57517,2025-01-02 00:00:00,2025-01-02 00:00:00,Corredores,Bretagne,"1,128.12",Abierto
5,S25-000002,4136574,2025-01-02 00:00:00,2025-01-02 00:00:00,Agentes,Ile-de-France,1204,Cerrado


In [7]:
pd.read_excel(ficheros[0], sheet_name="Detalle", header=None).tail(3)

,0,1,2,3,4,5,6,7
711,S25-000708,2241216,2025-01-13 00:00:00,2025-01-31 00:00:00,Agentes,Rhone-Alpes,"3,740.58",Abierto
712,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
713,TOTAL,NaN,NaN,NaN,NaN,NaN,"1,127,711.41",NaN


Problemas a resolver:
1. Tres filas de título antes de la cabecera → `header=3`.
2. Una fila vacía y una fila `TOTAL` al final → hay que quitarlas.
3. Una hoja `Resumen` con el total que, según el sistema, debería sumar el detalle → **la usaremos para cuadrar**.

Comparemos ahora enero y julio:

In [8]:
ene = pd.read_excel(ficheros[0], sheet_name="Detalle", header=3)
jul = pd.read_excel(ficheros[6], sheet_name="Detalle", header=3)
print("Enero:", ene.columns.tolist())
print("Julio:", jul.columns.tolist())
jul.head(3)

Enero: ['Nº Siniestro', 'Nº Póliza', 'Fecha ocurrencia', 'Fecha declaración', 'Canal', 'Región', 'Importe (€)', 'Estado']
Julio: ['N_SINIESTRO', 'N_POLIZA', 'F_OCURRENCIA', 'F_DECLARACION', 'CANAL', 'REGION', 'IMPORTE_EUR', 'ESTADO']


,N_SINIESTRO,N_POLIZA,F_OCURRENCIA,F_DECLARACION,CANAL,REGION,IMPORTE_EUR,ESTADO
0,S25-011801,"4,012,477.00",25/06/2025,01/07/2025,Directo,Ile-de-France,"1.204,00",CERRADO
1,S25-011802,"1,014,620.00",25/06/2025,01/07/2025,Corredores,Corse,"1.128,12",CERRADO
2,S25-011803,"5,032,837.00",17/06/2025,01/07/2025,Agentes,Ile-de-France,"1.204,00",CERRADO


El nuevo sistema (julio) usa **otros nombres de columna**, las **fechas vienen como texto** `dd/mm/aaaa` y los **importes como texto con formato español** (`1.204,00`). Es el típico problema que hoy se arregla a mano cada mes.

### Paso 1: un diccionario que traduce nombres de columna

In [9]:
MAPA_COLUMNAS = {
    "Nº Siniestro": "id_siniestro",         "N_SINIESTRO": "id_siniestro",
    "Nº Póliza": "id_poliza",               "N_POLIZA": "id_poliza",
    "Fecha ocurrencia": "fecha_ocurrencia", "F_OCURRENCIA": "fecha_ocurrencia",
    "Fecha declaración": "fecha_declaracion", "F_DECLARACION": "fecha_declaracion",
    "Canal": "canal",                       "CANAL": "canal",
    "Región": "region",                     "REGION": "region",
    "Importe (€)": "importe",               "IMPORTE_EUR": "importe",
    "Estado": "estado",                     "ESTADO": "estado",
}
jul.rename(columns=MAPA_COLUMNAS).columns.tolist()

['id_siniestro',
 'id_poliza',
 'fecha_ocurrencia',
 'fecha_declaracion',
 'canal',
 'region',
 'importe',
 'estado']

### ✏️ Ejercicio 1.1 · Importes con formato español
Escribe la función `a_numero(serie)` que convierta textos como `'1.234,56'` en el número `1234.56`. Si la serie ya es numérica, devuélvela tal cual.

Pista: `serie.str.replace(".", "", regex=False)` quita los puntos de miles; después cambia la coma por un punto y aplica `pd.to_numeric(..., errors="coerce")`.

In [10]:
def a_numero(serie):
    """'1.234,56' -> 1234.56 ; deja igual las series ya numéricas."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie.astype(float)
    texto = serie.astype(str).str.strip()
    texto = texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(texto, errors="coerce")

In [11]:
prueba = pd.Series(["1.234,56", "602,00", "-301,50", "4.075.400,56"])
comprobar(lambda: a_numero(prueba), pd.Series([1234.56, 602.0, -301.5, 4075400.56]),
          "Primero quita los puntos y después cambia la coma por punto (el orden importa)")

✅ ¡Correcto!


In [12]:
# 🛟 Red de seguridad: si tu función todavía no funciona, usamos la de referencia para poder seguir
try:
    assert np.allclose(a_numero(prueba), [1234.56, 602.0, -301.5, 4075400.56])
except Exception:
    from curso_seguros.cierre import a_numero
    print("🛟 Usando a_numero de referencia (curso_seguros.cierre) para continuar")

### Paso 2: una función que normaliza cualquier bordereau
Juntamos todo en una función. **Una sola función para los 12 meses**: si el mes que viene cambia algo, se arregla en un único sitio.

In [13]:
def a_fecha(serie):
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie
    return pd.to_datetime(serie, format="%d/%m/%Y", errors="coerce")


def normalizar_bordereau(ruta):
    """Lee un bordereau mensual (formato antiguo o nuevo) y lo estandariza."""
    df = pd.read_excel(ruta, sheet_name="Detalle", header=3)
    df = df.rename(columns=MAPA_COLUMNAS)
    df = df[df["id_siniestro"].notna() & (df["id_siniestro"] != "TOTAL")].copy()   # fuera vacías y TOTAL
    df["id_poliza"] = pd.to_numeric(df["id_poliza"], errors="coerce").astype("Int64")
    df["importe"] = a_numero(df["importe"])
    df["fecha_ocurrencia"] = a_fecha(df["fecha_ocurrencia"])
    df["fecha_declaracion"] = a_fecha(df["fecha_declaracion"])
    df["canal"] = df["canal"].astype(str).str.strip().str.title()      # ' AGENTES' -> 'Agentes'
    df["estado"] = df["estado"].astype(str).str.strip().str.title()
    df["mes"] = int(Path(ruta).stem[-2:])
    return df.reset_index(drop=True)


normalizar_bordereau(ficheros[6]).head()

,id_siniestro,id_poliza,fecha_ocurrencia,fecha_declaracion,canal,region,importe,estado,mes
0,S25-011801,4012477,2025-06-25,2025-07-01,Directo,Ile-de-France,"1,204.00",Cerrado,7
1,S25-011802,1014620,2025-06-25,2025-07-01,Corredores,Corse,"1,128.12",Cerrado,7
2,S25-011803,5032837,2025-06-17,2025-07-01,Agentes,Ile-de-France,"1,204.00",Cerrado,7
3,S25-011804,1001463,2025-06-28,2025-07-01,Agentes,Ile-de-France,"1,128.00",Cerrado,7
4,S25-011805,5032859,2025-07-01,2025-07-01,Bancaseguros,Rhone-Alpes,"1,204.00",Abierto,7


### ✏️ Ejercicio 1.2 · Consolidar los 12 meses
Aplica `normalizar_bordereau` a todos los `ficheros` y únelos con `pd.concat` en un único DataFrame `siniestros_brutos`.

Pista: crea una lista vacía, añade (`append`) cada DataFrame en un bucle y al final `pd.concat(lista, ignore_index=True)`.

In [14]:
partes = []
for f in ficheros:
    partes.append(normalizar_bordereau(f))
siniestros_brutos = pd.concat(partes, ignore_index=True)
siniestros_brutos.shape

(25551, 9)

In [15]:
from curso_seguros.cierre import normalizar_bordereau as _nb
_n = sum(len(_nb(f)[0]) for f in ficheros)
comprobar(lambda: len(siniestros_brutos), _n, "¿Has recorrido los 12 ficheros y usado pd.concat?")
comprobar(lambda: siniestros_brutos["importe"].dtype.kind, "f", "El importe debe ser numérico: ¿has aplicado a_numero?")

✅ ¡Correcto!
✅ ¡Correcto!


In [16]:
# 🛟 Red de seguridad
if not isinstance(siniestros_brutos, pd.DataFrame):
    siniestros_brutos = pd.concat([normalizar_bordereau(f) for f in ficheros], ignore_index=True)
    print("🛟 Usando la solución de referencia para continuar")

---
# Bloque 2 · Validación y control

El cierre no termina cuando "sale un número", sino cuando sabemos que **el número es correcto**. Cada control responde a una pregunta que hoy se revisa a ojo:

| Control | Pregunta |
|---|---|
| Duplicados | ¿Algún siniestro aparece dos veces? |
| Integridad referencial | ¿Todas las pólizas existen en cartera? |
| Rangos | ¿Hay importes negativos o absurdos? |
| Cuadre | ¿El detalle suma lo que dice el resumen del sistema? |
| Valores válidos | ¿Los canales son los 4 que existen? |

In [17]:
cartera = cargar_cartera()

dup = siniestros_brutos[siniestros_brutos.duplicated("id_siniestro", keep=False)]
print("Filas duplicadas:", siniestros_brutos.duplicated("id_siniestro").sum())
dup.sort_values("id_siniestro").head(4)

Filas duplicadas: 61


,id_siniestro,id_poliza,fecha_ocurrencia,fecha_declaracion,canal,region,importe,estado,mes
1301,S25-001302,2127038,2025-02-09,2025-02-14,Agentes,Lorraine,"1,705.16",Cerrado,2
1325,S25-001302,2127038,2025-02-09,2025-02-14,Agentes,Lorraine,"1,705.16",Cerrado,2
1477,S25-001477,1010436,2025-02-18,2025-02-18,Corredores,Languedoc-Roussillon,"1,128.12",Cerrado,2
1527,S25-001477,1010436,2025-02-18,2025-02-18,Corredores,Languedoc-Roussillon,"1,128.12",Cerrado,2


In [18]:
huerfanos = siniestros_brutos[~siniestros_brutos["id_poliza"].isin(cartera["id_poliza"])]
print("Siniestros con póliza inexistente:", len(huerfanos))
huerfanos[["id_siniestro", "id_poliza", "canal", "importe", "mes"]].head()

Siniestros con póliza inexistente: 36


,id_siniestro,id_poliza,canal,importe,mes
86,S25-000087,94082603,Directo,"1,204.00",1
108,S25-000109,91080579,Directo,713.34,1
234,S25-000235,93211248,Agentes,"1,204.00",1
777,S25-000778,95095921,Directo,"2,659.56",2
1105,S25-001106,93074291,Directo,"1,204.00",2


Las pólizas inexistentes empiezan por **9** seguido de un número válido: parece un error de tecleo en origen. **No los borramos** (el coste es real): los marcamos y se reportan al área de siniestros.

In [19]:
negativos = siniestros_brutos[siniestros_brutos["importe"] < 0]
negativos[["id_siniestro", "importe", "estado", "mes"]]

,id_siniestro,importe,estado,mes
4114,S25-003305-R,-102.12,Recobro,3
4115,S25-003536-R,"-1,243.28",Recobro,3
22157,S25-020728-R,-156.12,Recobro,10
22158,S25-021540-R,-301.00,Recobro,10


Son **recobros** (dinero que recuperamos de otra aseguradora). Son legítimos, pero hay que vigilarlos: un aviso, no un error.

### El cuadre contra la hoja Resumen

In [20]:
siniestros = siniestros_brutos.drop_duplicates("id_siniestro").copy()     # tratamiento: quitar duplicados

filas = []
for f in ficheros:
    resumen = pd.read_excel(f, sheet_name="Resumen").set_index("Concepto")["Valor"]
    mes = int(f.stem[-2:])
    detalle = siniestros[siniestros["mes"] == mes]
    filas.append({"mes": mes,
                  "n_resumen": resumen.iloc[0], "n_detalle": len(detalle),
                  "importe_resumen": resumen.iloc[1], "importe_detalle": detalle["importe"].sum()})
cuadre = pd.DataFrame(filas)
cuadre["diferencia"] = cuadre["importe_detalle"] - cuadre["importe_resumen"]
cuadre

,mes,n_resumen,n_detalle,importe_resumen,importe_detalle,diferencia
0,1,708.00,708,"1,127,711.41","1,127,711.41",0.00
1,2,"1,395.00",1395,"3,077,082.95","3,077,082.95",0.00
2,3,"2,007.00",2007,"7,796,494.64","7,796,494.64",0.00
3,4,"2,316.00",2316,"4,634,024.55","4,634,024.55",0.00
4,5,"2,690.00",2690,"6,858,504.88","6,858,504.88",-0.00
5,6,"2,686.00",2686,"4,985,795.65","4,985,795.65",0.00
6,7,"2,899.00",2899,"5,816,172.66","5,816,172.66",0.00
7,8,"2,668.00",2668,"6,632,866.34","6,632,866.34",0.00
8,9,"2,434.00",2433,"4,521,734.68","4,518,530.68","-3,204.00"
9,10,"2,303.00",2303,"4,944,529.20","4,944,529.20",0.00


### ✏️ Ejercicio 2.1
Añade a `cuadre` una columna `resultado` con `"OK"` si la diferencia de importe es menor que 1 céntimo en valor absoluto y `"KO"` si no. ¿Qué mes no cuadra?

Pista: `np.where(cuadre["diferencia"].abs() < 0.01, "OK", "KO")`

In [21]:
cuadre["resultado"] = np.where(cuadre["diferencia"].abs() < 0.01, "OK", "KO")
cuadre[["mes", "diferencia", "resultado"]]

,mes,diferencia,resultado
0,1,0.00,OK
1,2,0.00,OK
2,3,0.00,OK
3,4,0.00,OK
4,5,-0.00,OK
5,6,0.00,OK
6,7,0.00,OK
7,8,0.00,OK
8,9,"-3,204.00",KO
9,10,0.00,OK


In [22]:
comprobar(lambda: cuadre.loc[cuadre["resultado"] == "KO", "mes"].tolist(), [9],
          "Compara el valor ABSOLUTO de la diferencia con 0.01")

✅ ¡Correcto!


**Septiembre no cuadra:** en el detalle falta un siniestro de 3.204 € que sí aparece en el resumen. Sin este control, el cierre de septiembre habría infravalorado los costes.

### Todos los controles en una tabla (lo que irá a la hoja "Controles")
El módulo `curso_seguros.controles` tiene estos mismos controles empaquetados como funciones reutilizables:

In [23]:
from curso_seguros import controles as ctl

tabla_controles = ctl.ejecutar_controles([
    ctl.control_sin_duplicados(siniestros_brutos, "id_siniestro"),
    ctl.control_integridad(siniestros, "id_poliza", cartera["id_poliza"], "cartera"),
    ctl.control_rango(siniestros, "importe", minimo=0, aviso=True),
    ctl.control_sin_nulos(siniestros, ["id_siniestro", "id_poliza", "importe", "canal"]),
    ctl.control_cuadre("Cuadre importe anual vs Resumen", siniestros["importe"].sum(),
                       cuadre["importe_resumen"].sum()),
])
tabla_controles

,control,resultado,detalle
0,Sin duplicados en 'id_siniestro',KO,61 filas duplicadas
1,'id_poliza' existe en cartera,KO,36 registros sin correspondencia
2,"'importe' en rango [0, None]",AVISO,4 valores fuera de rango
3,Campos obligatorios informados,OK,sin nulos
4,Cuadre importe anual vs Resumen,KO,"calculado=58,180,088.56 | esperado=58,183,292...."


### ⭐ La versión profesional: `pandera`
`pandera` permite declarar **cómo deben ser los datos** (un "contrato") y validarlos de una vez. Con `lazy=True` informa de **todos** los fallos, no solo del primero.

In [24]:
import pandera.pandas as pa

esquema = pa.DataFrameSchema({
    "id_siniestro": pa.Column(unique=True),
    "id_poliza": pa.Column(checks=pa.Check(lambda s: s.isin(cartera["id_poliza"]), name="existe_en_cartera")),
    "importe": pa.Column(float, checks=[pa.Check.ge(0), pa.Check.le(5_000_000)]),
    "canal": pa.Column(checks=pa.Check.isin(["Agentes", "Corredores", "Directo", "Bancaseguros"])),
    "fecha_declaracion": pa.Column(checks=pa.Check.ge(pd.Timestamp("2025-01-01"))),
})
try:
    esquema.validate(siniestros_brutos, lazy=True)
except pa.errors.SchemaErrors as e:
    print(e.failure_cases.groupby(["column", "check"]).size())

column        check                      
id_poliza     existe_en_cartera               36
id_siniestro  field_uniqueness               122
importe       greater_than_or_equal_to(0)      4
dtype: int64


### `logging`: dejar rastro de lo que pasó
En un proceso automático nadie mira la pantalla. El `logging` deja un registro con fecha y hora (en el script se guarda en `ejecucion.log`).

In [25]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("cierre")
log.info("Siniestros cargados: %d", len(siniestros))
log.warning("Mes 9 no cuadra con el resumen: diferencia %.2f €", cuadre.loc[cuadre["mes"] == 9, "diferencia"].iloc[0])

2026-10-08 16:09:26,548 | INFO | Siniestros cargados: 25490


2026-10-08 16:09:26,549 | WARNING | Mes 9 no cuadra con el resumen: diferencia -3204.00 €


---
# Bloque 3 · Real vs presupuesto

### 3.1 El presupuesto: de formato ancho a formato largo
Los humanos leemos mejor el formato **ancho** (meses en columnas) y los ordenadores calculan mejor con el **largo** (una fila por canal-concepto-mes). `melt` es el "anular dinamización" de Power Query.

In [26]:
ppto_ancho = pd.read_excel(DATA / "entrada" / "presupuesto_2025.xlsx", header=2)
ppto_ancho.head(4)

,canal,concepto,Ene,Feb,Mar,Abr,May,Jun,Jul,Ago,Sep,Oct,Nov,Dic,Total
0,Agentes,Primas devengadas,1828383,2299273,3170687,3420323,3743113,3799155,4050538,3782489,3411346,3145861,2561939,1877607,37090714
1,Agentes,Coste de siniestros,1243301,1563506,2156067,2325820,2545317,2583426,2754366,2572093,2319715,2139186,1742119,1276773,25221689
2,Agentes,Comisiones,274258,344891,475603,513049,561467,569873,607581,567373,511702,471879,384291,281641,5563608
3,Agentes,Gastos de estructura,357705,357705,357705,357705,357705,357705,357705,357705,357705,357705,357705,357705,4292460


### ✏️ Ejercicio 3.1
Convierte `ppto_ancho` a formato largo en `ppto` con columnas `canal`, `concepto`, `mes_txt` e `importe`. Antes, quita la columna `Total` (si no, la tratarías como un mes más).

Pista: `ppto_ancho.drop(columns="Total").melt(id_vars=["canal", "concepto"], var_name="mes_txt", value_name="importe")`

In [27]:
ppto = ppto_ancho.drop(columns="Total").melt(id_vars=["canal", "concepto"], var_name="mes_txt",
                                             value_name="importe")
ppto.head()

,canal,concepto,mes_txt,importe
0,Agentes,Primas devengadas,Ene,1828383
1,Agentes,Coste de siniestros,Ene,1243301
2,Agentes,Comisiones,Ene,274258
3,Agentes,Gastos de estructura,Ene,357705
4,Bancaseguros,Primas devengadas,Ene,448778


In [28]:
comprobar(lambda: (len(ppto), round(ppto["importe"].sum())), (16 * 12, round(ppto_ancho["Total"].sum())),
          "Deben quedar 16 filas × 12 meses = 192 filas, y la suma debe coincidir con la columna Total")

✅ ¡Correcto!


In [29]:
# 🛟 Red de seguridad
if not isinstance(ppto, pd.DataFrame) or "mes_txt" not in ppto.columns:
    ppto = ppto_ancho.drop(columns="Total").melt(id_vars=["canal", "concepto"], var_name="mes_txt", value_name="importe")
    print("🛟 Usando la solución de referencia para continuar")

In [30]:
MESES_CORTOS = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
ppto["mes"] = ppto["mes_txt"].map({m: i + 1 for i, m in enumerate(MESES_CORTOS)})
ppto.sample(3, random_state=1)

,canal,concepto,mes_txt,importe,mes
44,Directo,Primas devengadas,Mar,1525220,3
69,Bancaseguros,Coste de siniestros,May,533475,5
161,Agentes,Coste de siniestros,Nov,1742119,11


### 3.2 Ingresos reales: prima devengada por mes
Igual que en la sesión 2, pero mes a mes: prima anual × días cubiertos **dentro del mes** / 365. La función `devengo_mensual` recorre los 12 meses:

In [31]:
from curso_seguros.cierre import devengo_mensual
import inspect
print(inspect.getsource(devengo_mensual))

def devengo_mensual(cartera: pd.DataFrame, anio: int = 2025) -> pd.DataFrame:
    """Prima devengada y pólizas en vigor por canal y mes.
    prima devengada del mes = prima anual × días cubiertos en el mes / 365."""
    filas = []
    for mes in range(1, 13):
        ini = pd.Timestamp(anio, mes, 1)
        fin = ini + pd.offsets.MonthBegin(1)
        dias = (cartera["fecha_vencimiento"].clip(upper=fin) - cartera["fecha_efecto"].clip(lower=ini)
                ).dt.days.clip(lower=0)
        g = pd.DataFrame({"canal": cartera["canal"], "prima": cartera["prima_anual"] * dias / 365,
                          "vigor": (dias > 0).astype(int)}).groupby("canal")[["prima", "vigor"]].sum()
        for canal, fila in g.iterrows():
            filas.append({"mes": mes, "canal": canal, "prima_devengada": fila["prima"],
                          "polizas_en_vigor": int(fila["vigor"])})
    return pd.DataFrame(filas)



In [32]:
devengo = devengo_mensual(cartera)
devengo.head()

,mes,canal,prima_devengada,polizas_en_vigor
0,1,Agentes,"1,847,121.84",119496
1,1,Bancaseguros,"434,289.76",27078
2,1,Corredores,"1,084,484.63",72255
3,1,Directo,"844,219.02",55401
4,2,Agentes,"2,308,874.76",148138


### 3.3 Reparto de gastos de estructura (contabilidad analítica)
Cada centro de coste se reparte entre canales con **su propia clave**:

In [33]:
parametros = pd.read_excel(DATA / "entrada" / "parametros" / "parametros_2025.xlsx", sheet_name=None)
parametros["Claves_reparto"]

,centro_coste,gasto_anual_presupuestado,clave_reparto
0,Tramitación de siniestros,3600000,n_siniestros
1,Tecnología,2400000,n_polizas
2,Marketing digital,1500000,solo_directo
3,Administración y finanzas,1300000,prima_devengada
4,Dirección comercial,1100000,prima_devengada


Ejemplo a mano para **un** centro y **un** mes: Tramitación de siniestros de septiembre, repartida por número de siniestros.

In [34]:
gastos = pd.read_excel(DATA / "entrada" / "gastos_estructura_2025.xlsx")
gasto_sep = gastos[(gastos["mes"] == "2025-09") & (gastos["centro_coste"] == "Tramitación de siniestros")]["importe"].sum()
n_sin_sep = siniestros[siniestros["mes"] == 9].groupby("canal").size()
peso = n_sin_sep / n_sin_sep.sum()
pd.DataFrame({"n_siniestros": n_sin_sep, "peso": peso, "gasto_asignado": gasto_sep * peso})

,n_siniestros,peso,gasto_asignado
canal,,,
Agentes,1063,0.44,"130,112.63"
Bancaseguros,225,0.09,"27,540.30"
Corredores,633,0.26,"77,480.05"
Directo,512,0.21,"62,669.49"


La función `reparto_gastos` hace lo mismo para **todos** los centros y meses a la vez (con `merge` y `groupby().transform`):

In [35]:
from curso_seguros.cierre import reparto_gastos

drivers = (devengo.rename(columns={"polizas_en_vigor": "n_polizas"})
           .merge(siniestros.groupby(["mes", "canal"], as_index=False).agg(n_siniestros=("id_siniestro", "size")),
                  on=["mes", "canal"], how="left").fillna({"n_siniestros": 0}))
reparto = reparto_gastos(gastos, parametros["Claves_reparto"], drivers)
reparto.pivot_table(index="centro_coste", columns="canal", values="gasto_asignado", aggfunc="sum").round(0)

canal,Agentes,Bancaseguros,Corredores,Directo
centro_coste,,,,
Administración y finanzas,"593,934.00","140,286.00","349,949.00","284,661.00"
Dirección comercial,"541,482.00","127,894.00","319,043.00","259,516.00"
Marketing digital,0.00,0.00,0.00,"1,778,688.00"
Tecnología,"1,170,246.00","265,045.00","707,722.00","556,112.00"
Tramitación de siniestros,"1,693,863.00","387,103.00","1,034,069.00","838,081.00"


### 3.4 La cuenta de resultados y la comparación

In [36]:
from curso_seguros.cierre import cuenta_resultados_mensual, tabla_resultados, real_vs_presupuesto

real = cuenta_resultados_mensual(siniestros, devengo, parametros["Comisiones"], reparto)
MES = 9
ytd = list(range(1, MES + 1))
tabla_resultados(real, ytd)

concepto,Primas devengadas,Coste de siniestros,Comisiones,Gastos de estructura,Margen técnico,Siniestralidad %,Ratio gastos+comisiones %,Ratio combinado %
canal,,,,,,,,
Agentes,"28,777,830.85","22,701,013.37","4,316,674.63","3,061,538.12","-1,301,395.27",0.79,0.26,1.05
Bancaseguros,"6,804,312.63","3,131,132.30","1,360,862.53","698,396.50","1,613,921.31",0.46,0.30,0.76
Corredores,"16,963,562.04","9,855,654.46","3,053,441.17","1,848,828.07","2,205,638.35",0.58,0.29,0.87
Directo,"13,919,537.59","9,759,383.63","278,390.75","2,889,596.57","992,166.64",0.70,0.23,0.93
Total,"66,465,243.12","45,447,183.76","9,009,369.07","8,498,359.25","3,510,331.04",0.68,0.26,0.95


In [37]:
rvp = real_vs_presupuesto(real, ppto, ytd)
rvp[rvp["concepto"] == "Margen técnico"]

,canal,concepto,Real,Presupuesto,Desviación,Desviación %,Valoración
4,Agentes,Margen técnico,"-1,301,395.27",1796554,"-3,097,949.27",-1.72,Desfavorable
9,Corredores,Margen técnico,"2,205,638.35",872643,"1,332,995.35",1.53,Favorable
14,Directo,Margen técnico,"992,166.64",2448953,"-1,456,786.36",-0.59,Desfavorable
19,Bancaseguros,Margen técnico,"1,613,921.31",635209,"978,712.31",1.54,Favorable
24,Total,Margen técnico,"3,510,331.04",5753359,"-2,243,027.96",-0.39,Desfavorable


In [38]:
rvp[rvp["canal"] == "Directo"]

,canal,concepto,Real,Presupuesto,Desviación,Desviación %,Valoración
10,Directo,Primas devengadas,"13,919,537.59",14298426,"-378,888.41",-0.03,Desfavorable
11,Directo,Coste de siniestros,"9,759,383.63",10008899,"-249,515.37",-0.02,Favorable
12,Directo,Comisiones,"278,390.75",285968,"-7,577.25",-0.03,Favorable
13,Directo,Gastos de estructura,"2,889,596.57",1554606,"1,334,990.57",0.86,Desfavorable
14,Directo,Margen técnico,"992,166.64",2448953,"-1,456,786.36",-0.59,Desfavorable


### 💬 Para discutir
El canal **Directo** tiene una desviación de gastos de estructura del +86 %. ¿Ha gastado de más? No exactamente:
- En el **presupuesto**, los gastos se repartieron en proporción a la prima.
- En el **real**, *Marketing digital* se imputa al 100 % a Directo (clave `solo_directo`).

**La desviación es, en buena parte, un efecto del criterio de reparto**, no de gestión. Es la conversación típica de FP&A que el dato bien estructurado permite tener (y explicar) en minutos.

---
# Bloque 4 · Salidas automáticas

### 4.1 Excel con formato (`xlsxwriter`)
`df.to_excel()` vuelca los datos sin formato. Con `xlsxwriter` controlamos formatos numéricos, anchos, colores, paneles inmovilizados e incluso gráficos nativos de Excel.

In [39]:
resultado = tabla_resultados(real, ytd)
ruta_xlsx = OUTPUT / "demo" / "resultado_simple.xlsx"
ruta_xlsx.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(ruta_xlsx, engine="xlsxwriter") as xw:
    resultado.to_excel(xw, sheet_name="Resultado", startrow=2)
    libro, hoja = xw.book, xw.sheets["Resultado"]
    euros = libro.add_format({"num_format": '#,##0 "€";[Red]-#,##0 "€"'})
    pct = libro.add_format({"num_format": "0.0%"})
    titulo = libro.add_format({"bold": True, "font_size": 14, "font_color": "#1F4E79"})
    hoja.write("A1", f"Cuenta técnica acumulada a septiembre 2025", titulo)
    hoja.set_column("A:A", 16)
    hoja.set_column("B:F", 18, euros)
    hoja.set_column("G:I", 14, pct)
    hoja.freeze_panes(3, 1)
    # semáforo en el ratio combinado: rojo si > 100 %
    rojo = libro.add_format({"bg_color": "#FFC7CE", "font_color": "#9C0006"})
    hoja.conditional_format("I4:I8", {"type": "cell", "criteria": ">", "value": 1, "format": rojo})
print("Guardado:", ruta_xlsx)

Guardado: /content/curso-python-finanzas/output/demo/resultado_simple.xlsx


La versión completa (varias hojas, semáforos, hoja de controles y gráfico nativo) está en `curso_seguros.informes.escribir_excel_cierre`. Úsala como plantilla para tus propios informes.

### 4.2 PDF de una página (`fpdf2`)

In [40]:
from fpdf import FPDF
import matplotlib

fuentes = Path(matplotlib.get_data_path()) / "fonts" / "ttf"   # fuente con € y tildes
pdf = FPDF()
pdf.add_page()
pdf.add_font("DejaVu", "", str(fuentes / "DejaVuSans.ttf"))
pdf.set_font("DejaVu", "", 14)
pdf.cell(0, 10, "Cierre de septiembre 2025 · Autos", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("DejaVu", "", 10)
tot = resultado.loc["Total"]
pdf.multi_cell(0, 6, f"Primas devengadas: {tot['Primas devengadas']:,.0f} €\n"
                     f"Margen técnico: {tot['Margen técnico']:,.0f} €\n"
                     f"Ratio combinado: {tot['Ratio combinado %']:.1%}")
pdf.output(str(OUTPUT / "demo" / "resumen_simple.pdf"))
print("PDF generado")

2026-10-08 16:09:28,070 | INFO | maxp pruned


2026-10-08 16:09:28,071 | INFO | cmap pruned


2026-10-08 16:09:28,073 | INFO | kern dropped


2026-10-08 16:09:28,074 | INFO | post pruned


2026-10-08 16:09:28,075 | INFO | FFTM dropped


...

2026-10-08 16:09:28,108 | INFO | OS/2 Unicode ranges pruned: [0, 1, 33]


2026-10-08 16:09:28,109 | INFO | OS/2 CodePage ranges pruned: [0]


2026-10-08 16:09:28,110 | INFO | glyf pruned


2026-10-08 16:09:28,111 | INFO | name pruned


PDF generado


> ⚠️ Las fuentes básicas de PDF (Helvetica, Times) **no tienen el símbolo €**. Por eso cargamos la fuente DejaVu, que viene incluida con matplotlib.

### 4.3 Un correo por responsable de canal (modo simulacro)
El patrón "**combinar correspondencia**": un bucle que, para cada canal, genera su Excel filtrado y su correo con el adjunto. Los correos se guardan como `.eml`: se abren con doble clic en Outlook y se revisan **antes** de enviar nada.

In [41]:
from curso_seguros.cierre import ejecutar_cierre
from curso_seguros.informes import escribir_excel_cierre
from curso_seguros.correo import construir_correo, guardar_eml

res = ejecutar_cierre(MES)          # todo lo anterior en una llamada
DESTINATARIOS = {"Agentes": "dir.agentes@ejemplo.com", "Corredores": "dir.corredores@ejemplo.com",
                 "Directo": "dir.directo@ejemplo.com", "Bancaseguros": "dir.bancaseguros@ejemplo.com"}

carpeta = OUTPUT / "demo" / "correos"
for canal, email in DESTINATARIOS.items():
    adjunto = escribir_excel_cierre(res, OUTPUT / "demo" / f"cierre_09_{canal}.xlsx", canal=canal)
    r = res["resultado_ytd"].loc[canal]
    cuerpo = (f"Hola,\n\nAdjunto el cierre de septiembre del canal {canal}.\n"
              f"Margen técnico acumulado: {r['Margen técnico']:,.0f} € "
              f"(ratio combinado {r['Ratio combinado %']:.1%}).\n\nUn saludo,\nFP&A Autos")
    msg = construir_correo("fpa-autos@ejemplo.com", [email], f"Cierre 2025-09 · {canal}", cuerpo, [adjunto])
    print("✉️ ", guardar_eml(msg, carpeta, f"correo_{canal}").name)

2026-10-08 16:09:28,593 | INFO | Bordereau 01: 708 siniestros, 1127711.41 €


2026-10-08 16:09:28,751 | INFO | Bordereau 02: 1395 siniestros, 3077082.95 €


2026-10-08 16:09:28,975 | INFO | Bordereau 03: 2007 siniestros, 7796494.64 €


2026-10-08 16:09:29,238 | INFO | Bordereau 04: 2316 siniestros, 4634024.55 €


2026-10-08 16:09:29,534 | INFO | Bordereau 05: 2690 siniestros, 6858504.88 €


2026-10-08 16:09:29,893 | INFO | Bordereau 06: 2686 siniestros, 4985795.65 €


2026-10-08 16:09:30,228 | INFO | Bordereau 07: 2899 siniestros, 5816172.66 €


2026-10-08 16:09:30,554 | INFO | Bordereau 08: 2668 siniestros, 6632866.34 €


2026-10-08 16:09:30,844 | INFO | Bordereau 09: 2433 siniestros, 4518530.68 €


✉️  correo_Agentes.eml
✉️  correo_Corredores.eml
✉️  correo_Directo.eml
✉️  correo_Bancaseguros.eml


> **¿Y el envío real?** Desde Colab no se envía correo corporativo (Google bloquea el puerto SMTP 25 y, además, Office 365 está retirando el envío SMTP con usuario y contraseña). En la empresa se hace con **Microsoft Graph** u **Outlook** (`win32com` en Windows), configurado con IT. La lógica de construir el mensaje es la misma; solo cambia la última línea.

---
# Bloque 5 · Del notebook al proceso

Un notebook es ideal para **explorar y explicar**. Un proceso que se ejecuta cada mes necesita un **script**: se lanza con un comando, recibe parámetros y deja log. Mira el script del curso:

In [42]:
print(Path("scripts/run_report.py").read_text(encoding="utf-8")[:2500])

"""
Cierre mensual automático del ramo Autos.

Ejecuta, en un solo paso:
  1. Carga y valida los bordereaux de siniestros hasta el mes indicado.
  2. Calcula la cuenta técnica real (primas, siniestros, comisiones, gastos).
  3. Compara contra presupuesto (mes y acumulado).
  4. Genera Excel completo + un Excel por canal + PDF resumen.
  5. Prepara un correo por responsable de canal (simulacro: ficheros .eml).

Uso:
    python scripts/run_report.py                    # usa el periodo de config/params.yaml
    python scripts/run_report.py --periodo 2025-06
    python scripts/run_report.py --periodo 2025-12 --sin-correos

Programarlo (Windows):  Programador de tareas → Crear tarea básica → Acción
"Iniciar un programa" → python.exe con argumentos "scripts\\run_report.py".
"""
from __future__ import annotations

import argparse
import logging
import sys
from pathlib import Path

import yaml

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ / "src"))

from curso_seguros

### Configuración fuera del código
Lo que cambia (periodo, destinatarios, rutas) va en `config/params.yaml`, no en el código. Así cambiar un destinatario no exige tocar Python.

In [43]:
print(Path("config/params.yaml").read_text(encoding="utf-8"))

# Parámetros del cierre mensual. Cambiar aquí, NO en el código.
periodo: "2025-09"            # mes a cerrar (AAAA-MM). Se puede sobrescribir con --periodo

salidas:
  carpeta: "output"           # se crea output/<periodo>/
  excel: true
  pdf: true
  correos: true

correo:
  modo_simulacro: true        # true = guarda .eml; false = enviar (requiere configurar IT)
  remitente: "fpa-autos@ejemplo.com"
  destinatarios:              # responsables de canal (direcciones ficticias)
    Agentes: ["dir.agentes@ejemplo.com"]
    Corredores: ["dir.corredores@ejemplo.com"]
    Directo: ["dir.directo@ejemplo.com"]
    Bancaseguros: ["dir.bancaseguros@ejemplo.com"]
  copia_direccion: ["direccion.financiera@ejemplo.com"]

controles:
  detener_si_ko: false        # true = no generar informes si algún control falla



### Estructura del proyecto

```
curso-python-finanzas/
├── config/          ← parámetros (YAML)
├── data/
│   ├── entrada/     ← lo que nos mandan (nunca se modifica a mano)
│   └── ...
├── notebooks/       ← exploración y formación
├── output/          ← lo que generamos (no se sube a Git)
├── scripts/         ← procesos ejecutables (run_report.py)
├── src/curso_seguros/ ← funciones reutilizables (la "lógica de negocio")
├── requirements.txt ← librerías necesarias
└── README.md        ← cómo se usa
```

**Regla:** la lógica vive en `src/` y la usan tanto los notebooks como los scripts. **Una sola versión de la verdad.**

### Git en 15 minutos
Git es el "control de cambios" de una carpeta entera: guarda quién cambió qué, cuándo y por qué, y permite volver atrás.

| Concepto | Analogía | Comando | En GitHub Desktop |
|---|---|---|---|
| Repositorio | La carpeta del proyecto con historial | `git clone <url>` | *File → Clone repository* |
| Commit | Una versión guardada con comentario | `git add . && git commit -m "Añade control de cuadre"` | Escribir el resumen → *Commit to main* |
| Push / Pull | Subir / bajar cambios del servidor | `git push` / `git pull` | *Push origin* / *Fetch origin* |
| Rama | Una copia para probar sin romper nada | `git switch -c nuevo-informe` | *Current branch → New branch* |
| Pull request | Pedir que revisen y aprueben tus cambios | (en la web de GitHub) | *Create Pull Request* |

**Qué NO se sube a Git** (`.gitignore`): datos confidenciales, la carpeta `output/`, contraseñas y ficheros `.env`. Tampoco las **salidas de los notebooks**, que pueden contener cifras; para eso está `nbstripout`.

### Programar la ejecución
- **Windows:** Programador de tareas → *Crear tarea básica* → mensual, día 3 → *Iniciar un programa*: `python scripts\run_report.py`.
- **Servidor o nube:** cron, Azure Data Factory, Airflow o GitHub Actions, según lo que tenga IT.

### ✏️ Ejercicio 5.1 · Cierre de diciembre
Ejecuta el script para el periodo **2025-12** (sin correos) y guarda en `margen_anual` el margen técnico total del año que aparece en la hoja "Resultado YTD" del Excel generado.

Pista: usa `subprocess.run` como en el bloque 0, y después `pd.read_excel(..., sheet_name="Resultado YTD", header=2, index_col=0)`.

In [44]:
r = subprocess.run([sys.executable, "scripts/run_report.py", "--periodo", "2025-12", "--sin-correos"],
                   capture_output=True, text=True)
excel_dic = pd.read_excel(OUTPUT / "2025-12" / "cierre_2025-12.xlsx", sheet_name="Resultado YTD",
                          header=2, index_col=0)
margen_anual = excel_dic.loc["Total", "Margen técnico"]
print(f"Margen técnico 2025: {margen_anual:,.0f} €")

Margen técnico 2025: 2,976,102 €


In [45]:
comprobar(lambda: margen_anual, tabla_resultados(ejecutar_cierre(12)["real_largo"], list(range(1, 13))).loc["Total", "Margen técnico"],
          "Lee la hoja 'Resultado YTD' con header=2 e index_col=0 y busca la fila Total", rtol=1e-6)

2026-10-08 16:09:38,869 | INFO | Bordereau 01: 708 siniestros, 1127711.41 €


2026-10-08 16:09:39,030 | INFO | Bordereau 02: 1395 siniestros, 3077082.95 €


2026-10-08 16:09:39,264 | INFO | Bordereau 03: 2007 siniestros, 7796494.64 €


2026-10-08 16:09:39,506 | INFO | Bordereau 04: 2316 siniestros, 4634024.55 €


2026-10-08 16:09:39,883 | INFO | Bordereau 05: 2690 siniestros, 6858504.88 €


...

2026-10-08 16:09:41,291 | INFO | Bordereau 09: 2433 siniestros, 4518530.68 €


2026-10-08 16:09:41,618 | INFO | Bordereau 10: 2303 siniestros, 4944529.20 €


2026-10-08 16:09:41,949 | INFO | Bordereau 11: 1914 siniestros, 5050876.94 €


2026-10-08 16:09:42,149 | INFO | Bordereau 12: 1471 siniestros, 2737498.66 €


✅ ¡Correcto!


---
## Resumen de la sesión
- **Una función por tarea** (`normalizar_bordereau`, `a_numero`…), aplicada en bucle a N ficheros.
- **Controles explícitos**: duplicados, integridad, rangos y cuadres, con una hoja de controles en el informe.
- **Formato largo** para calcular (`melt`) y **ancho** para presentar (`pivot_table`).
- El reparto de gastos es una decisión de negocio que **explica desviaciones**.
- Excel, PDF y correos se generan **desde los mismos datos**, sin copiar y pegar.
- Notebook para explorar; **script + config + Git** para producir.

**Próxima sesión:** ¿el margen negativo de Agentes es estructural o mala suerte? Estadística para dar **rangos** en vez de números puntuales.